# Member 4 — KG-Supported Symbolic Reasoning, Hybrid Integration & LLM Explanations

**Google Colab | Run in order, one stage at a time**

This notebook covers your implementation, **not Member 3's ML training**.

1. **Stage 0 — Setup and verify:** Upload merged dataset and Member 2's two KG files; verify the original reference labels.
2. **Stage 1 — KG integration:** Validate event/user/device context and join verified KG evidence **before** reasoning.
3. **Stage 2 — Symbolic reasoning:** Run **R1–R11**. R7 requires KG same-user confirmation; R6 also requires **time gap ≤10 minutes**. Save evidence, risk and priority.
4. **Stage 3 — Hybrid integration:** Once available, load Member 3's **held-out** Random Forest predictions and apply the agreed **higher-risk** fusion policy.
5. **Stage 4 — Explanation:** Generate an evidence-grounded LLM explanation (optional API) or local **non-LLM** preview.
6. **Stage 5 — Evaluation:** Compare ML, symbolic and hybrid classifications against finalized rule-derived labels; review explanation faithfulness.
7. **Stage 6 — Export:** Save deliverables.

**Important:** The original `risk_score` and `risk_level` are finalized reference labels and are never used as inputs to symbolic reasoning, hybrid fusion or the LLM. They were created by a scoring rubric, not independently verified incident outcomes.


## Stage 0 — Setup and files

**What to do:** Upload all three CSVs: `MergedDataset(1).csv`, `kg_relationships.csv`, and `kg_scenario_context.csv`. You do **not** need the preprocessed dataset for Member 4's component.

The merged file has no scenario ID, so we create `S0001`, `S0002`, ... based on its original row order. We verify the KG's event IDs against that mapping before using the KG. **Do not reorder the merged CSV before running this notebook.** If the merged file changes, regenerate and re-share `scenario_id_mapping.csv`.


In [21]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from IPython.display import display

RISK_LEVELS = ["Low", "Medium", "High"]
RISK_RANK = {label: index for index, label in enumerate(RISK_LEVELS)}
MERGED_FILE = "newData.csv"
KG_RELATIONSHIPS_FILE = "kg_relationships.csv"
KG_CONTEXT_FILE = "kg_scenario_context.csv"

# Member 2's files are now required for KG-supported R6 and R7.
USE_ML_FILE = True       # Change to True after Member 3 provides held-out predictions
USE_LLM_API = True      # False uses a clearly labeled local template
USE_KG_FILE = True
MAX_EXPLANATIONS = 160
REVIEW_SAMPLE_SIZE = 30
REVIEW_RANDOM_STATE = 42
LLM_MODEL = "gpt-4o-mini"  # Change only if you have API access to another model
print("Configuration ready: KG is required before symbolic reasoning.")

Configuration ready: KG is required before symbolic reasoning.


In [5]:
# Upload the merged dataset and BOTH Member 2 files in Colab, if not already present.
needed = [MERGED_FILE, KG_RELATIONSHIPS_FILE, KG_CONTEXT_FILE]
missing_files = [f for f in needed if not Path(f).exists()]
if missing_files:
    try:
        from google.colab import files
        print("Upload these files:", missing_files)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Place these files in the working directory: {missing_files}")
if any(not Path(f).exists() for f in needed):
    raise FileNotFoundError(f"Still missing: {[f for f in needed if not Path(f).exists()]}")

merged = pd.read_csv(MERGED_FILE)
merged.columns = merged.columns.str.strip()
# Member 1's merged file has no scenario_id. Member 2 uses S0001... in its original row order.
# Check the mapping using event IDs in Stage 1 before trusting it.
if "scenario_id" not in merged.columns:
    merged.insert(0, "scenario_id", [f"S{i:04d}" for i in range(1, len(merged) + 1)])
if merged.scenario_id.isna().any() or merged.scenario_id.duplicated().any():
    raise ValueError("Merged scenario IDs must be unique and non-null")
print("Merged dataset:", merged.shape)
display(merged.head(3))

Upload these files: ['newData.csv', 'kg_relationships.csv', 'kg_scenario_context.csv']


Saving kg_scenario_context.csv to kg_scenario_context.csv
Saving newData.csv to newData.csv
Saving kg_relationships.csv to kg_relationships.csv
Merged dataset: (800, 34)


,scenario_id,event_id_physical,user_id,physical_timestamp,location,location_type,access_result,event_number,event_id_cyber,device_id_x,...,is_weekend,normal_start_hour,normal_end_hour,after_hours,restricted_location,access_denied,time_gap_minutes,severity_numeric,risk_score,risk_level
0,S0001,P0001,U050,2026-04-06 03:44:30,Research Lab,Restricted,Granted,1,C0001,D050,...,0,8,18,1,1,0,10.416667,1,3.0,Medium
1,S0002,P0002,U045,2026-03-17 09:42:37,Meeting Room,Normal,Granted,2,C0002,D045,...,0,0,23,0,0,0,14.733333,0,0.0,Low
2,S0003,P0003,U030,2026-03-19 04:16:14,Server Room,Restricted,Denied,3,C0003,D030,...,0,8,17,1,1,1,8.200000,0,4.0,Medium


### Step 0.1 — Validate the finalized labeling (do not alter it)

The colleague's actual notebook computes the risk score as follows: `after_hours` (+1), `restricted_location` (+1), `access_denied` (+1), `severity_numeric` (+0/1/2), `failed_logins >= 5` (+1), `failed_logins >= 10` (another +1), `sensitive_files >= 10` (+1), `usb_connected == 1` (+1), `time_gap_minutes <= 10` (+1). **Low = score ≤2; Medium = 3–5; High = score >5.**

The next cell **checks**, rather than rewrites, these existing columns. If it fails, investigate the input before proceeding.
**Important:** Member 1 updated the label-generation cutoff to **10 minutes**. Upload her newly regenerated merged dataset with updated `risk_score` and `risk_level`; the older 30-minute dataset will fail validation. R6 also uses 10 minutes. Do not modify labels within this notebook.


In [6]:
required = ["user_id", "event_id_physical", "event_id_cyber", "device_id_x",
            "after_hours", "restricted_location", "access_denied", "failed_logins",
            "sensitive_files", "usb_connected", "severity_numeric", "time_gap_minutes",
            "risk_score", "risk_level"]
missing = [c for c in required if c not in merged.columns]
if missing:
    raise ValueError(f"Missing merged-data columns: {missing}")
if merged[required].isna().any().any():
    raise ValueError("Required columns contain missing values; inspect the data before continuing.")

expected_score = (merged.after_hours + merged.restricted_location + merged.access_denied
                  + merged.severity_numeric + (merged.failed_logins >= 5).astype(int)
                  + (merged.failed_logins >= 10).astype(int)
                  + (merged.sensitive_files >= 10).astype(int)
                  + (merged.usb_connected == 1).astype(int)
                  + (merged.time_gap_minutes.between(0, 10, inclusive="both")).astype(int))
expected_level = expected_score.map(lambda x: "Low" if x <= 2 else "Medium" if x <= 5 else "High")
assert np.allclose(expected_score, merged.risk_score), "Risk score differs from colleague's script."
assert expected_level.equals(merged.risk_level),"Risk level differs from colleague's script."
print("Verified: existing risk scores and labels match colleague's script.")
print(merged.risk_level.value_counts().reindex(RISK_LEVELS).to_string())

# Export stable scenario-to-event mapping for Members 2 and 3.
key_cols = ["scenario_id", "user_id", "event_id_physical", "event_id_cyber", "device_id_x"]
merged[key_cols].to_csv("scenario_id_mapping.csv", index=False)
print("Saved scenario_id_mapping.csv — share this mapping with teammates.")

Verified: existing risk scores and labels match colleague's script.
risk_level
Low       456
Medium    217
High      127
Saved scenario_id_mapping.csv — share this mapping with teammates.


## Stage 1 — Load and verify Member 2's KG evidence **before** symbolic reasoning

**Run this stage after validating Member 1's existing labels.** Upload `kg_relationships.csv` and `kg_scenario_context.csv` with the merged CSV in Stage 0.

- `kg_relationships.csv` contains `scenario_id`, `kg_same_user`, `kg_same_device`, and `kg_evidence`.
- `kg_scenario_context.csv` contains the physical/cyber event IDs, timestamps, location and activity details. It does **not** have a scenario ID, so we join it using the **physical and cyber event ID pair**, never by row position.
- We check that the event IDs, user, device, and key context match the merged dataset. Mismatches stop the notebook instead of silently treating unrelated events as linked.
- **R6/R7 use `kg_same_user` explicitly**. If the KG does not confirm linkage, those rules do not trigger. A failed or missing KG match stops execution for review.

The original `risk_score` and `risk_level` are used only for reference/evaluation, never for symbolic decisions.

In [7]:
# 1. Load KG relationship flags, one record per scenario.
kg_rel = pd.read_csv(KG_RELATIONSHIPS_FILE, dtype={"scenario_id": str})
kg_ctx = pd.read_csv(KG_CONTEXT_FILE)
kg_rel.columns = kg_rel.columns.str.strip()
kg_ctx.columns = kg_ctx.columns.str.strip()

required_rel = {"scenario_id", "kg_same_user", "kg_same_device", "kg_evidence"}
required_ctx = {"physical_event_id", "cyber_event_id", "user_id", "device_id",
                "physical_timestamp", "cyber_timestamp", "location", "location_type",
                "access_result", "after_hours", "cyber_severity", "failed_logins",
                "sensitive_files", "usb_connected", "time_gap_minutes"}
if required_rel - set(kg_rel.columns) or required_ctx - set(kg_ctx.columns):
    raise ValueError(f"Missing KG fields: relationships={required_rel-set(kg_rel.columns)}, "
                     f"context={required_ctx-set(kg_ctx.columns)}")
if kg_rel.scenario_id.isna().any() or kg_rel.scenario_id.duplicated().any():
    raise ValueError("KG relationship scenario_id must be unique and non-null")
if kg_ctx.duplicated(["physical_event_id", "cyber_event_id"]).any():
    raise ValueError("KG context contains duplicate physical/cyber event pairs")

def strict_bool(value):
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if str(value).strip().lower() in {"true", "1", "yes"}: return True
    if str(value).strip().lower() in {"false", "0", "no"}: return False
    raise ValueError(f"Invalid or missing KG boolean: {value!r}")

for c in ["kg_same_user", "kg_same_device"]:
    kg_rel[c] = kg_rel[c].map(strict_bool)

# 2. Verify scenario-ID mapping using event IDs, not row order alone.
kg_ctx = kg_ctx.rename(columns={
    "physical_event_id": "event_id_physical", "cyber_event_id": "event_id_cyber",
    "user_id": "kg_context_user_id", "device_id": "kg_context_device_id",
    "physical_timestamp": "kg_physical_timestamp", "cyber_timestamp": "kg_cyber_timestamp",
    "location": "kg_location", "location_type": "kg_location_type",
    "access_result": "kg_access_result", "after_hours": "kg_after_hours",
    "cyber_severity": "kg_cyber_severity", "failed_logins": "kg_failed_logins",
    "sensitive_files": "kg_sensitive_files", "usb_connected": "kg_usb_connected",
    "time_gap_minutes": "kg_time_gap_minutes"})

# For every scenario ID, confirm that KG evidence names the same two events.
kg_pairs = kg_rel[["scenario_id", "kg_evidence"]].merge(
    merged[["scenario_id", "event_id_physical", "event_id_cyber"]],
    on="scenario_id", how="right", validate="one_to_one", indicator=True)
if (kg_pairs._merge != "both").any() or len(kg_rel) != len(merged):
    raise ValueError("KG relationship IDs do not match merged scenarios")
if not kg_pairs.apply(lambda r: str(r.event_id_physical) in str(r.kg_evidence)
                      and str(r.event_id_cyber) in str(r.kg_evidence), axis=1).all():
    raise ValueError("KG evidence event IDs disagree with scenario mapping")

# 3. Join context by the two event IDs, then flags by scenario_id.
enriched_merged = merged.merge(kg_ctx, on=["event_id_physical", "event_id_cyber"],
                               how="left", validate="one_to_one", indicator=True)
if (enriched_merged._merge != "both").any() or len(kg_ctx) != len(merged):
    raise ValueError("KG context event pairs do not match merged event pairs")
enriched_merged = enriched_merged.drop(columns="_merge").merge(
    kg_rel, on="scenario_id", how="left", validate="one_to_one")
if enriched_merged[["kg_same_user", "kg_same_device", "kg_evidence"]].isna().any().any():
    raise ValueError("Missing KG relationship evidence after join")

# 4. Validate that KG context refers to the same user/device and facts.
checks = {
    "user_id": enriched_merged.user_id.astype(str).eq(enriched_merged.kg_context_user_id.astype(str)),
    "device_id": enriched_merged.device_id_x.astype(str).eq(enriched_merged.kg_context_device_id.astype(str)),
    "location": enriched_merged.location.astype(str).eq(enriched_merged.kg_location.astype(str)),
    "severity": enriched_merged.severity.astype(str).str.lower().eq(enriched_merged.kg_cyber_severity.astype(str).str.lower()),
    "failed_logins": enriched_merged.failed_logins.eq(enriched_merged.kg_failed_logins),
    "sensitive_files": enriched_merged.sensitive_files.eq(enriched_merged.kg_sensitive_files),
    "usb_connected": enriched_merged.usb_connected.astype(int).eq(enriched_merged.kg_usb_connected.map(strict_bool).astype(int)),
    "after_hours": enriched_merged.after_hours.astype(int).eq(enriched_merged.kg_after_hours.map(strict_bool).astype(int)),
    "time_gap_minutes": np.isclose(enriched_merged.time_gap_minutes,
                                   enriched_merged.kg_time_gap_minutes, atol=0.001),
    "physical_timestamp": pd.to_datetime(enriched_merged.physical_timestamp).eq(pd.to_datetime(enriched_merged.kg_physical_timestamp)),
    "cyber_timestamp": pd.to_datetime(enriched_merged.cyber_timestamp).eq(pd.to_datetime(enriched_merged.kg_cyber_timestamp)),
}
for name, matches in checks.items():
    if not np.asarray(matches).all():
        raise ValueError(f"KG/merged mismatch for {name}: {int((~np.asarray(matches)).sum())} rows")
# This dataset has one device per merged scenario; the KG flag is retained as evidence.
print(f"KG validated: {len(enriched_merged)} scenarios, "
      f"{enriched_merged.kg_same_user.sum()} same-user links, "
      f"{enriched_merged.kg_same_device.sum()} same-device links")
display(enriched_merged[["scenario_id", "event_id_physical", "event_id_cyber",
                         "kg_same_user", "kg_same_device", "kg_location", "kg_evidence"]].head(5))

KG validated: 800 scenarios, 800 same-user links, 800 same-device links


,scenario_id,event_id_physical,event_id_cyber,kg_same_user,kg_same_device,kg_location,kg_evidence
0,S0001,P0001,C0001,True,True,Research Lab,S0001: P0001 -> C0001; User U050 PERFORMED P00...
1,S0002,P0002,C0002,True,True,Meeting Room,S0002: P0002 -> C0002; User U045 PERFORMED P00...
2,S0003,P0003,C0003,True,True,Server Room,S0003: P0003 -> C0003; User U030 PERFORMED P00...
3,S0004,P0004,C0004,True,True,Meeting Room,S0004: P0004 -> C0004; User U014 PERFORMED P00...
4,S0005,P0005,C0005,True,True,Server Room,S0005: P0005 -> C0005; User U046 PERFORMED P00...


## Stage 2 — Apply all 11 symbolic rules to KG-enriched scenarios

**What this does:** R1, R2 and R8 detect physical anomalies; R3–R5 and R9–R11 detect cyber anomalies. **R7** triggers when both anomalies exist and the KG confirms the same user. **R6** also requires a time gap of **10 minutes or less**.

Risk policy: no anomaly → **Low**; one domain → **Medium**; linked anomalies in both domains → **High**. `priority` is **Time-critical** for R6, **Elevated** for R7, otherwise **Normal**. It does not create a fourth risk label.

Every rule stores its own factual evidence. Reference `risk_score` and `risk_level` are deliberately excluded from the rule engine.
**Updated agreement:** Both R6 and Member 1’s revised risk-scoring formula use **≤10 minutes**. Stage 0 validates the *new* labels; old datasets based on 30 minutes must not be used for final evaluation.


In [8]:
def apply_symbolic_rules(row):
    """Apply R1-R11 using features and verified KG links; never consult reference labels."""
    rules, facts = [], []
    def add(rule_id, inference, evidence):
        rules.append(rule_id)
        facts.append({"rule": rule_id, "inference": inference, "facts": evidence})

    # Physical rules
    if row["after_hours"] == 1 and row["restricted_location"] == 1:
        add("R1", "PhysicalAnomaly", ["after_hours=1", "restricted_location=1",
                                       f"location={row['kg_location']}"])
    if row["access_denied"] == 1 and row["restricted_location"] == 1:
        add("R2", "PhysicalAnomaly", ["access_denied=1", "restricted_location=1",
                                       f"location={row['kg_location']}"])
    if row["after_hours"] == 1 and row["access_denied"] == 1:
        add("R8", "PhysicalAnomaly", ["after_hours=1", "access_denied=1"])

    # Cyber rules: thresholds aligned with Member 1's underlying indicators
    if row["failed_logins"] >= 5 and row["sensitive_files"] >= 10:
        add("R3", "CyberAnomaly", [f"failed_logins={row['failed_logins']}",
                                    f"sensitive_files={row['sensitive_files']}"])
    if row["usb_connected"] == 1 and row["sensitive_files"] >= 10:
        add("R4", "CyberAnomaly", ["usb_connected=1", f"sensitive_files={row['sensitive_files']}"])
    if row["severity_numeric"] == 2 and row["failed_logins"] >= 5:
        add("R5", "CyberAnomaly", ["severity=High", f"failed_logins={row['failed_logins']}"])
    if row["severity_numeric"] == 2 and row["sensitive_files"] >= 10:
        add("R9", "CyberAnomaly", ["severity=High", f"sensitive_files={row['sensitive_files']}"])
    if row["severity_numeric"] == 2 and row["usb_connected"] == 1:
        add("R10", "CyberAnomaly", ["severity=High", "usb_connected=1"])
    if row["usb_connected"] == 1 and row["failed_logins"] >= 5:
        add("R11", "CyberAnomaly", ["usb_connected=1", f"failed_logins={row['failed_logins']}"])

    physical = bool(set(rules) & {"R1", "R2", "R8"})
    cyber = bool(set(rules) & {"R3", "R4", "R5", "R9", "R10", "R11"})
    same_user = bool(row["kg_same_user"])  # Explicit KG confirmation, not just a non-null user_id
    if physical and cyber and same_user:
        add("R7", "ElevatedCyberPhysicalRisk", ["physical_anomaly=true", "cyber_anomaly=true",
             "kg_same_user=true", f"physical_timestamp={row['kg_physical_timestamp']}",
             f"cyber_timestamp={row['kg_cyber_timestamp']}",
             f"kg_relationship={row['kg_evidence']}"])
        if row["time_gap_minutes"] <= 10:
            add("R6", "ElevatedCyberPhysicalRisk", ["physical_anomaly=true", "cyber_anomaly=true",
                 "kg_same_user=true", f"time_gap_minutes={row['time_gap_minutes']:.3f}",
                 "threshold_minutes=10"])

    risk = "High" if "R7" in rules else "Medium" if physical or cyber else "Low"
    priority = "Time-critical" if "R6" in rules else "Elevated" if "R7" in rules else "Normal"
    return pd.Series({"symbolic_risk": risk, "priority": priority,
                      "triggered_rules": rules, "supporting_evidence": facts,
                      "physical_anomaly": physical, "cyber_anomaly": cyber,
                      "cross_domain_rule": "R7" in rules})

symbolic_details = enriched_merged.apply(apply_symbolic_rules, axis=1)
symbolic = pd.concat([enriched_merged[["scenario_id"]], symbolic_details], axis=1)
assert len(symbolic) == len(enriched_merged)
print("Symbolic risk distribution:")
print(symbolic.symbolic_risk.value_counts().reindex(RISK_LEVELS, fill_value=0).to_string())
print("Triggered rules frequency:")
print(pd.Series([r for rules in symbolic.triggered_rules for r in rules]).value_counts().to_string())
display(symbolic[["scenario_id", "symbolic_risk", "priority", "triggered_rules"]].head(8))

Symbolic risk distribution:
symbolic_risk
Low       441
Medium    236
High      123
Triggered rules frequency:
R1     252
R3     164
R11    138
R4     127
R7     123
R5     117
R2     116
R9     105
R8     101
R10     86
R6      36


,scenario_id,symbolic_risk,priority,triggered_rules
0,S0001,Medium,Normal,[R1]
1,S0002,Low,Normal,[]
2,S0003,Medium,Normal,"[R1, R2, R8]"
3,S0004,Medium,Normal,"[R3, R5, R9]"
4,S0005,High,Time-critical,"[R1, R2, R8, R3, R4, R5, R9, R10, R11, R7, R6]"
5,S0006,Medium,Normal,[R1]
6,S0007,Low,Normal,[]
7,S0008,High,Elevated,"[R1, R3, R4, R5, R9, R10, R11, R7]"


### Stage 2.1 — Inspect and export KG-supported symbolic results

Run this after Stage 2. Inspect examples from each risk class, including the **actual evidence** and KG linkage. The export is your symbolic output for the next stages.

In [9]:
for level in RISK_LEVELS:
    subset = symbolic[symbolic.symbolic_risk == level].head(2)
    print(f"\n{level}: {len(symbolic[symbolic.symbolic_risk == level])} total")
    display(subset[["scenario_id", "priority", "triggered_rules", "supporting_evidence"]])

# Join evidence into the symbolic output BEFORE the hybrid/LLM stages.
enriched = symbolic.merge(
    enriched_merged[["scenario_id", "kg_same_user", "kg_same_device", "kg_evidence",
                     "kg_location", "kg_physical_timestamp", "kg_cyber_timestamp",
                     "kg_time_gap_minutes"]], on="scenario_id", how="left", validate="one_to_one")
enriched["kg_review_required"] = (~enriched.kg_same_user) | (~enriched.kg_same_device)
if enriched.kg_review_required.any():
    print("Review KG-unlinked scenarios:")
    display(enriched.loc[enriched.kg_review_required,
                         ["scenario_id", "kg_same_user", "kg_same_device"]].head(10))

symbolic_export = enriched.copy()
symbolic_export["triggered_rules"] = symbolic_export.triggered_rules.map(json.dumps)
symbolic_export["supporting_evidence"] = symbolic_export.supporting_evidence.map(json.dumps)
symbolic_export.to_csv("member4_symbolic_results.csv", index=False)
print("Saved member4_symbolic_results.csv")


Low: 441 total


,scenario_id,priority,triggered_rules,supporting_evidence
1,S0002,Normal,[],[]
6,S0007,Normal,[],[]



Medium: 236 total


,scenario_id,priority,triggered_rules,supporting_evidence
0,S0001,Normal,[R1],"[{'rule': 'R1', 'inference': 'PhysicalAnomaly'..."
2,S0003,Normal,"[R1, R2, R8]","[{'rule': 'R1', 'inference': 'PhysicalAnomaly'..."



High: 123 total


,scenario_id,priority,triggered_rules,supporting_evidence
4,S0005,Time-critical,"[R1, R2, R8, R3, R4, R5, R9, R10, R11, R7, R6]","[{'rule': 'R1', 'inference': 'PhysicalAnomaly'..."
7,S0008,Elevated,"[R1, R3, R4, R5, R9, R10, R11, R7]","[{'rule': 'R1', 'inference': 'PhysicalAnomaly'..."


Saved member4_symbolic_results.csv


In [10]:
if USE_KG_FILE:
    KG_FILE = KG_RELATIONSHIPS_FILE
    if not Path(KG_FILE).exists():
        from google.colab import files
        print(f"Upload {KG_FILE}")
        files.upload()
    kg = pd.read_csv(KG_FILE, dtype={"scenario_id": str})
    required_kg = ["scenario_id", "kg_same_user", "kg_same_device", "kg_evidence"]
    if any(c not in kg.columns for c in required_kg):
        raise ValueError(f"KG output requires: {required_kg}")
    if kg.scenario_id.isna().any() or kg.scenario_id.duplicated().any():
        raise ValueError("KG scenario_id must be non-null and unique")
    def parse_bool(v):
        if str(v).strip().lower() in ("true", "1", "yes"): return True
        if str(v).strip().lower() in ("false", "0", "no"): return False
        raise ValueError(f"Invalid KG boolean: {v}")
    kg["kg_same_user"] = kg.kg_same_user.map(parse_bool)
    kg["kg_same_device"] = kg.kg_same_device.map(parse_bool)
    if not kg.scenario_id.isin(symbolic.scenario_id).all():
        raise ValueError("KG contains unknown scenario IDs")
    enriched = symbolic.merge(kg, on="scenario_id", how="left", validate="one_to_one", indicator="kg_merge_status")
    enriched["kg_review_required"] = (
        enriched.kg_merge_status.ne("both") |
        enriched.kg_same_user.ne(True).fillna(True) |
        enriched.kg_same_device.ne(True).fillna(True)
    )
    print("KG records joined:", int(enriched.kg_merge_status.eq("both").sum()))
    print("Missing KG records:", int(enriched.kg_merge_status.ne("both").sum()))
    print("Contradictions / missing requiring review:", int(enriched.kg_review_required.sum()))
    if enriched.kg_review_required.any():
        display(enriched.loc[enriched.kg_review_required,
                 ["scenario_id", "kg_same_user", "kg_same_device", "kg_evidence"]].head())
else:
    enriched = symbolic.copy()
    enriched["kg_same_user"] = pd.NA
    enriched["kg_same_device"] = pd.NA
    enriched["kg_evidence"] = "KG verification not supplied"
    enriched["kg_review_required"] = True
    print("KG verification deferred: all scenarios require KG review before final integration.")

# Export symbolic + KG output even while ML integration is pending.
export_symbolic = enriched.copy()
export_symbolic["triggered_rules"] = export_symbolic.triggered_rules.map(json.dumps)
export_symbolic["supporting_evidence"] = export_symbolic.supporting_evidence.map(json.dumps)
export_symbolic.to_csv("member4_symbolic_kg_results.csv", index=False)
print("Saved member4_symbolic_kg_results.csv")
if enriched.kg_review_required.any():
    print("WARNING: Some scenarios require relationship review; do not treat their results as final.")


KG records joined: 800
Missing KG records: 0
Contradictions / missing requiring review: 0
Saved member4_symbolic_kg_results.csv


## Stage 3 — Hybrid integration (Member 3's Random Forest)

**Do this when Member 3 sends held-out test predictions.** Set `USE_ML_FILE = True` in Stage 0, upload `ml_predictions.csv` containing `scenario_id` and `ml_prediction`, then rerun from Stage 0. Predictions must be for held-out test scenarios and use the same scenario IDs.

**Agreed fusion rule:** take the higher of the ML and symbolic risk (Low < Medium < High). Keep `agreement_status` to analyze disagreements. This policy may increase false alarms, so Stage 5 evaluates it.

Until then, leave `USE_ML_FILE = False`; no artificial predictions are used.

In [11]:
if USE_ML_FILE:
    ML_FILE = "ml_predictions.csv"
    if not Path(ML_FILE).exists():
        from google.colab import files
        print(f"Upload {ML_FILE}")
        files.upload()
    # ml = pd.read_csv(ML_FILE, dtype={"scenario_id": str})

    ml = pd.read_csv(ML_FILE)
    # Convert zero-based row indices to notebook scenario IDs
    ml["scenario_id"] = ml["scenario_id"].apply(
        lambda x: f"S{int(x):04d}"
    )

    if not {"scenario_id", "ml_prediction"}.issubset(ml.columns):
        raise ValueError("ML file needs scenario_id and ml_prediction")
    if ml.scenario_id.isna().any() or ml.scenario_id.duplicated().any():
        raise ValueError("ML scenario IDs must be unique and non-null")
    ml["ml_prediction"] = ml.ml_prediction.astype(str).str.strip().str.title()
    if not ml.ml_prediction.isin(RISK_LEVELS).all():
        raise ValueError("ML predictions must be Low, Medium, or High")
    if not ml.scenario_id.isin(merged.scenario_id).all():
        raise ValueError("Some ML scenario IDs do not exist in the merged dataset")
    hybrid = ml.merge(enriched, on="scenario_id", how="left", validate="one_to_one")
    if hybrid.symbolic_risk.isna().any():
        raise ValueError("ML and symbolic scenario IDs are not aligned")
    hybrid["final_hybrid_risk"] = hybrid.apply(
        lambda r: max([r["ml_prediction"], r["symbolic_risk"]], key=RISK_RANK.get), axis=1)
    hybrid["agreement_status"] = np.where(
        hybrid.ml_prediction == hybrid.symbolic_risk, "Agree", "Disagree")
    print(f"Real held-out ML predictions integrated: {len(hybrid)}")
    display(hybrid[["scenario_id", "ml_prediction", "symbolic_risk",
                    "final_hybrid_risk", "agreement_status"]].head(10))
else:
    hybrid = None
    print("Waiting for Member 3's held-out ML predictions. Continue to Stage 4 for a symbolic-only explanation preview.")

Upload ml_predictions.csv


Saving ml_predictions.csv to ml_predictions.csv
Real held-out ML predictions integrated: 160


,scenario_id,ml_prediction,symbolic_risk,final_hybrid_risk,agreement_status
0,S0064,Low,Low,Low,Agree
1,S0008,High,High,High,Agree
2,S0365,Low,Low,Low,Agree
3,S0383,High,High,High,Agree
4,S0748,Low,Low,Low,Agree
5,S0112,Low,Low,Low,Agree
6,S0590,Low,Low,Low,Agree
7,S0403,High,High,High,Agree
8,S0692,Low,Low,Low,Agree
9,S0658,Low,Low,Low,Agree


### Step 3.1 — Examine disagreements

Once Member 3's output is loaded, inspect cases where the Random Forest and symbolic rules disagree. Higher-risk selection is a **policy choice**, not a guarantee of higher accuracy. This table helps you explain the trade-off in the presentation.

In [12]:
if hybrid is not None:
    print(hybrid.agreement_status.value_counts().to_string())
    display(hybrid.loc[hybrid.agreement_status == "Disagree",
                       ["scenario_id", "ml_prediction", "symbolic_risk",
                        "final_hybrid_risk", "triggered_rules"]].head(12))
else:
    print("Skipped: no real ML predictions yet.")

agreement_status
Agree       153
Disagree      7


,scenario_id,ml_prediction,symbolic_risk,final_hybrid_risk,triggered_rules
15,S0644,High,Medium,High,"[R3, R4, R5, R9, R10, R11]"
23,S0504,Medium,High,High,"[R1, R9, R7]"
59,S0080,Medium,Low,Medium,[]
60,S0678,Low,Medium,Medium,[R1]
131,S0454,Low,Medium,Medium,[R3]
135,S0430,Medium,High,High,"[R1, R4, R7]"
158,S0735,Medium,High,High,"[R1, R4, R7]"


In [13]:
# Compare the types of disagreement
pd.crosstab(
    hybrid["ml_prediction"],
    hybrid["symbolic_risk"],
    rownames=["ML Prediction"],
    colnames=["Symbolic Risk"]
)

Symbolic Risk,High,Low,Medium
ML Prediction,,,
High,25,0,1
Low,0,88,2
Medium,3,1,40


## Stage 4 — Explain the fixed decision using an LLM (or test locally)

The LLM **does not classify**. It receives the fixed final risk, triggered rules, supporting facts and KG relationship evidence. The payload **excludes** the reference `risk_score`/`risk_level` and raw user/device identifiers. The optional API uses a hidden key prompt; the default local template is **not** an LLM explanation.

You can preview symbolic-only explanations before Member 3's predictions arrive. When ML is integrated, rerun this stage for hybrid explanations.

In [14]:
# Bring KG context into hybrid using verified event IDs

context_cols = [
    "event_id_physical",
    "event_id_cyber",
    "kg_location",
    "kg_time_gap_minutes"
]

# Join KG context to scenario IDs
kg_context_by_scenario = kg_pairs[
    ["scenario_id", "event_id_physical", "event_id_cyber"]
].merge(
    kg_ctx[context_cols],
    on=["event_id_physical", "event_id_cyber"],
    how="left",
    validate="one_to_one"
)

# Ensure one context record per scenario
assert kg_context_by_scenario["scenario_id"].is_unique

# Attach context to hybrid results
hybrid = hybrid.merge(
    kg_context_by_scenario[
        ["scenario_id", "kg_location", "kg_time_gap_minutes"]
    ],
    on="scenario_id",
    how="left",
    validate="one_to_one"
)

print(
    hybrid[
        ["scenario_id", "kg_location", "kg_time_gap_minutes"]
    ].head()
)

print(
    "Missing locations:",
    hybrid["kg_location"].isna().sum()
)

print(
    "Missing time gaps:",
    hybrid["kg_time_gap_minutes"].isna().sum()
)

  scenario_id     kg_location  kg_time_gap_minutes
0       S0064  Finance Office              23.5167
1       S0008    Storage Room              12.6000
2       S0365  Finance Office              11.1833
3       S0383    Storage Room              21.5500
4       S0748  General Office               6.8667
Missing locations: 0
Missing time gaps: 0


In [27]:
# Build a structured, evidence-only payload; reference risk_score/risk_level are excluded.
def make_payload(row, is_hybrid):
    evidence = row["supporting_evidence"]
    if isinstance(evidence, str): evidence = json.loads(evidence)
    rules = row["triggered_rules"]
    if isinstance(rules, str): rules = json.loads(rules)
    # KG evidence strings contain raw identifiers. The LLM only needs verified linkage,
    # and the rule-specific facts above (R7 evidence is sanitized below).
    safe_evidence = []
    for item in evidence:
        safe_item = dict(item)
        safe_item["facts"] = [f for f in item["facts"]
                              if not str(f).startswith(("kg_relationship=", "physical_timestamp=", "cyber_timestamp="))]
        safe_evidence.append(safe_item)
    result = {
        "assessment_type": "hybrid" if is_hybrid else "symbolic_only_preview",
        "final_risk": row["final_hybrid_risk"] if is_hybrid else row["symbolic_risk"],
        "symbolic_risk": row["symbolic_risk"], "priority": row["priority"],
        "triggered_rules": rules, "supporting_evidence": safe_evidence,
        "kg_same_user": bool(row["kg_same_user"]),
        "kg_same_device": bool(row["kg_same_device"]),
        "kg_location": row["kg_location"],
        "kg_time_gap_minutes": float(row["kg_time_gap_minutes"]),
    }
    if is_hybrid:
        result["ml_prediction"] = row["ml_prediction"]
        result["agreement_status"] = row["agreement_status"]
    return result

explanation_source = hybrid if hybrid is not None else enriched
is_hybrid = hybrid is not None
sample = explanation_source.copy()

assert len(sample) == MAX_EXPLANATIONS, (
    f"Expected {MAX_EXPLANATIONS} scenarios, found {len(sample)}"
)

sample["llm_payload"] = [make_payload(r, is_hybrid) for _, r in sample.iterrows()]
print("Example structured LLM input:")
print(json.dumps(sample.iloc[0].llm_payload, indent=2, ensure_ascii=False))

Example structured LLM input:
{
  "assessment_type": "hybrid",
  "final_risk": "Low",
  "symbolic_risk": "Low",
  "priority": "Normal",
  "triggered_rules": [],
  "supporting_evidence": [],
  "kg_same_user": true,
  "kg_same_device": true,
  "kg_location": "Finance Office",
  "kg_time_gap_minutes": 23.5167,
  "ml_prediction": "Low",
  "agreement_status": "Agree"
}


In [28]:
def local_explanation(payload):
    """No-API fallback: useful for pipeline testing; NOT an LLM-generated explanation."""
    facts = [f"{e['rule']} ({', '.join(e['facts'])})" for e in payload["supporting_evidence"]]
    reason = "; ".join(facts) if facts else "No configured anomaly combination was triggered"
    prefix = "Hybrid assessment" if payload["assessment_type"] == "hybrid" else "Symbolic-only preview"
    return f"{prefix}: {payload['final_risk']}. Supporting evidence: {reason}."

def generate_explanation(payload):
    if not USE_LLM_API:
        return local_explanation(payload)
    from getpass import getpass
    from openai import OpenAI
    if not hasattr(generate_explanation, "client"):
        generate_explanation.client = OpenAI(api_key=getpass("OpenAI API key (hidden): "))
    response = generate_explanation.client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": (
            "You are a cybersecurity analyst assistant explaining a fixed "
            "hybrid risk assessment. You are NOT a classifier. "
            "Never change the supplied final risk. "
            "First, state the final risk level. "
            "Then explain the triggered symbolic rules and their supporting facts. "
            "If ML and symbolic reasoning disagree, explicitly state both "
            "predictions and explain that the final risk is determined by "
            "selecting the higher of the two risk levels. "
            "Distinguish observed evidence from inferred risk. "
            "Do not invent events, causes, or supporting facts. "
            "Do not describe a location or time gap as suspicious unless "
            "the provided evidence supports that interpretation. "
            "If no symbolic rules were triggered, state this clearly. "
            "Write 2-4 concise, professional sentences."
            )},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}
        ])
    return response.choices[0].message.content.strip()

if USE_LLM_API:
    try:
        import openai
    except ImportError:
        %pip -q install openai
sample["explanation"] = sample.llm_payload.map(generate_explanation)
sample["explanation_method"] = "LLM API" if USE_LLM_API else "Local template (not LLM)"
display(sample[["scenario_id", "explanation_method", "explanation"]])

OpenAI API key (hidden): ··········


,scenario_id,explanation_method,explanation
0,S0064,LLM API,Final risk level: Low. \n\nNo symbolic rules w...
1,S0008,LLM API,The final risk level is High. The triggered sy...
2,S0365,LLM API,The final risk level is Low. No symbolic rules...
3,S0383,LLM API,The final risk level is High. The triggered sy...
4,S0748,LLM API,Final risk level: Low. \n\nNo symbolic rules w...
...,...,...,...
155,S0769,LLM API,The final risk level is Low. No symbolic rules...
156,S0173,LLM API,The final risk level is Low. No symbolic rules...
157,S0530,LLM API,The final risk level is Medium. The triggered ...
158,S0735,LLM API,The final risk level is High. The triggered sy...


In [29]:
for _, row in sample.iterrows():
    print("=" * 60)
    print("Scenario:", row["scenario_id"])
    print("Explanation:", row["explanation"])
    print()

Scenario: S0064
Explanation: Final risk level: Low. 

No symbolic rules were triggered, and there is no supporting evidence to indicate a higher risk. Both the machine learning prediction and symbolic reasoning agree on a low risk level, confirming the assessment. The observed evidence, including the same user and device, supports the inferred low risk.

Scenario: S0008
Explanation: The final risk level is High. The triggered symbolic rules include R1, indicating a physical anomaly due to after-hours access to a restricted location (Storage Room), and multiple cyber anomalies (R3, R4, R5, R9, R10, R11) related to failed logins, sensitive files, and USB connections, all with high severity. The evidence supports an elevated cyber-physical risk (R7) as both physical and cyber anomalies are present, and they involve the same user and device. The machine learning prediction also aligns with this assessment, confirming a High risk level.

Scenario: S0365
Explanation: The final risk level is 

## Stage 5 — Evaluation

**Only evaluate final hybrid accuracy on the same held-out scenarios used by Member 3.** `risk_level` is the unchanged **rubric-derived reference label**. Do not use it as an input to your rules, fusion decision, or LLM prompt.

We'll compare: (1) Random Forest, (2) symbolic-only, (3) final hybrid. Report accuracy, macro F1, classification report, confusion matrix, and the number of disagreements. In addition, compare High-risk recall and false escalations, because the higher-risk policy may increase both.

If Member 3's predictions are not yet available, this stage deliberately skips classification evaluation rather than generating misleading numbers.

In [30]:
if hybrid is None:
    print("Classification evaluation deferred until real held-out ML predictions are uploaded.")
else:
    from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                                 confusion_matrix, recall_score)
    evaluated = hybrid.merge(merged[["scenario_id", "risk_level"]],
                             on="scenario_id", how="left", validate="one_to_one")
    assert evaluated.risk_level.notna().all()
    summary = []
    for label, pred_col in [("Random Forest", "ml_prediction"),
                            ("Symbolic", "symbolic_risk"),
                            ("Hybrid", "final_hybrid_risk")]:
        y_true, y_pred = evaluated.risk_level, evaluated[pred_col]
        summary.append({"method": label, "accuracy": accuracy_score(y_true, y_pred),
                        "macro_f1": f1_score(y_true, y_pred, labels=RISK_LEVELS,
                                             average="macro", zero_division=0),
                        "high_recall": recall_score(y_true, y_pred, labels=["High"],
                                                    average="macro", zero_division=0),
                        "low_or_medium_escalated_to_high": int(((y_true != "High") &
                                                                (y_pred == "High")).sum())})
        print(f"\n--- {label} ---")
        print(classification_report(y_true, y_pred, labels=RISK_LEVELS, zero_division=0))
        print("Confusion matrix (rows=true, cols=pred; Low, Medium, High):")
        print(confusion_matrix(y_true, y_pred, labels=RISK_LEVELS))
    metrics = pd.DataFrame(summary)
    display(metrics)
    metrics.to_csv("member4_evaluation_metrics.csv", index=False)
    print("Saved member4_evaluation_metrics.csv")


--- Random Forest ---
              precision    recall  f1-score   support

         Low       0.99      0.98      0.98        91
      Medium       0.95      0.98      0.97        43
        High       1.00      1.00      1.00        26

    accuracy                           0.98       160
   macro avg       0.98      0.98      0.98       160
weighted avg       0.98      0.98      0.98       160

Confusion matrix (rows=true, cols=pred; Low, Medium, High):
[[89  2  0]
 [ 1 42  0]
 [ 0  0 26]]

--- Symbolic ---
              precision    recall  f1-score   support

         Low       1.00      0.98      0.99        91
      Medium       0.93      0.93      0.93        43
        High       0.89      0.96      0.93        26

    accuracy                           0.96       160
   macro avg       0.94      0.96      0.95       160
weighted avg       0.96      0.96      0.96       160

Confusion matrix (rows=true, cols=pred; Low, Medium, High):
[[89  2  0]
 [ 0 40  3]
 [ 0  1 25]]

--

,method,accuracy,macro_f1,high_recall,low_or_medium_escalated_to_high
0,Random Forest,0.98125,0.982981,1.000000,0
1,Symbolic,0.96250,0.948349,0.961538,3
2,Hybrid,0.96250,0.952976,1.000000,3


Saved member4_evaluation_metrics.csv


### Step 5.1 — Manual explanation faithfulness review

**Do not claim an LLM evaluation result from local-template previews.** When real LLM responses are generated, review a sample manually against the structured input. Check:

1. **Risk consistency:** Did it preserve the provided final risk?
2. **Evidence faithfulness:** Are every mentioned facts and relationships present in the payload?
3. **No hallucinations:** Did it avoid unsupported incidents, causal claims, or new indicators?
4. **Clarity:** Can a reader understand why the assessment was reached?

Enter 1 (pass) or 0 (fail) in the exported review template after manual inspection. Blank values mean **not yet reviewed**.

In [31]:
print("sample columns:")
print(sample.columns.tolist())

print("\nNumber of rows:", len(sample))

print("\nExplanation columns available:")
for name in ["sample", "explanation_source", "hybrid"]:
    if name in globals():
        df = globals()[name]
        print(name, [
            c for c in ["explanation", "explanation_method"]
            if c in df.columns
        ])

sample columns:
['scenario_id', 'ml_prediction', 'symbolic_risk', 'priority', 'triggered_rules', 'supporting_evidence', 'physical_anomaly', 'cyber_anomaly', 'cross_domain_rule', 'kg_same_user', 'kg_same_device', 'kg_evidence', 'kg_merge_status', 'kg_review_required', 'final_hybrid_risk', 'agreement_status', 'kg_location', 'kg_time_gap_minutes', 'llm_payload', 'explanation', 'explanation_method']

Number of rows: 160

Explanation columns available:
sample ['explanation', 'explanation_method']
explanation_source []
hybrid []


In [32]:
# Randomly select 30 explanations from the full generated set
review = sample.sample(
    n=REVIEW_SAMPLE_SIZE,
    random_state=REVIEW_RANDOM_STATE
).copy()

# Keep relevant fields for manual verification
review = review[
    [
        "scenario_id",
        "ml_prediction",
        "symbolic_risk",
        "final_hybrid_risk",
        "agreement_status",
        "triggered_rules",
        "supporting_evidence",
        "explanation_method",
        "explanation"
    ]
].copy()

# Manual evaluation criteria
for col in [
    "risk_consistent",
    "evidence_faithful",
    "no_hallucination",
    "clear_explanation"
]:
    review[col] = ""

review["review_notes"] = ""

# Save review template
review.to_csv(
    "member4_explanation_review_template.csv",
    index=False
)

print(f"Total explanations generated: {len(sample)}")
print(f"Randomly selected for review: {len(review)}")

display(review.head())

Total explanations generated: 160
Randomly selected for review: 30


,scenario_id,ml_prediction,symbolic_risk,final_hybrid_risk,agreement_status,triggered_rules,supporting_evidence,explanation_method,explanation,risk_consistent,evidence_faithful,no_hallucination,clear_explanation,review_notes
105,S0011,Medium,Medium,Medium,Agree,[R11],"[{'rule': 'R11', 'inference': 'CyberAnomaly', ...",LLM API,The final risk level is Medium. The triggered ...,,,,,
108,S0301,Low,Low,Low,Agree,[],[],LLM API,Final risk level: Low. \n\nNo symbolic rules w...,,,,,
141,S0741,Low,Low,Low,Agree,[],[],LLM API,The final risk level is Low. No symbolic rules...,,,,,
55,S0270,Medium,Medium,Medium,Agree,[R9],"[{'rule': 'R9', 'inference': 'CyberAnomaly', '...",LLM API,The final risk level is Medium. The triggered ...,,,,,
94,S0052,Low,Low,Low,Agree,[],[],LLM API,The final risk level is Low. No symbolic rules...,,,,,


## Stage 6 — Export deliverables

- `scenario_id_mapping.csv` — original merged-to-scenario mapping
- `member4_symbolic_results.csv` — R1–R11 results, risk, priority, KG-supported evidence
- `member4_hybrid_results.csv` — only after real held-out ML predictions are supplied
- `member4_evaluation_metrics.csv` — only after ML predictions are supplied
- `member4_explanations.csv` and `member4_explanation_review_template.csv`

**Methodology note:** Member 1's finalized labels were generated using scoring rules. Report evaluation as agreement with those rule-derived reference labels, not independently verified real-world incident accuracy. R6 and R7 explicitly depend on KG same-user evidence; R6 additionally requires ≤10 minutes.

In [33]:
if hybrid is not None:
    export_hybrid = hybrid.copy()
    export_hybrid["triggered_rules"] = export_hybrid.triggered_rules.map(json.dumps)
    export_hybrid["supporting_evidence"] = export_hybrid.supporting_evidence.map(json.dumps)
    export_hybrid.to_csv("member4_hybrid_results.csv", index=False)
    print("Saved member4_hybrid_results.csv")
else:
    print("No hybrid CSV yet: waiting for Member 3's test predictions.")

explanations_export = sample[["scenario_id", "explanation_method", "explanation"]].copy()
explanations_export.to_csv("member4_explanations.csv", index=False)
print("Saved member4_explanations.csv")
print("Notebook run complete through the currently available inputs.")

Saved member4_hybrid_results.csv
Saved member4_explanations.csv
Notebook run complete through the currently available inputs.


In [35]:


files_to_check = {
    "scenario_id_mapping.csv": 800,
    "member4_symbolic_results.csv": 800,
    "member4_symbolic_kg_results.csv": 800,
    "member4_hybrid_results.csv": 160,
    "member4_evaluation_metrics.csv": 3,
    "member4_explanations.csv": 160,
    "member4_explanation_review_template.csv": 30
}

for filename, expected in files_to_check.items():
    actual = len(pd.read_csv(filename))
    status = "OK" if actual == expected else "CHECK"
    print(f"{status}: {filename} — {actual}/{expected} rows")

OK: scenario_id_mapping.csv — 800/800 rows
OK: member4_symbolic_results.csv — 800/800 rows
OK: member4_symbolic_kg_results.csv — 800/800 rows
OK: member4_hybrid_results.csv — 160/160 rows
OK: member4_evaluation_metrics.csv — 3/3 rows
OK: member4_explanations.csv — 160/160 rows
OK: member4_explanation_review_template.csv — 30/30 rows
